# Protocolo compartido de desarrollo

Holdout aproximado 80/20, semilla 42, balance de presencia multilabel, objetos, tamaños y fuente. Los grupos sugeridos por nombres no se consideran verificados. El test oficial no se abre para diseñar la partición.

Preparado para Colab T4. Entrenamientos, evaluaciones y benchmarks desactivados. Configurar las mismas rutas persistentes en todos los notebooks. Nunca consultar test durante selección.

La creación de splits está activada por solicitud del equipo. Al ejecutar todas las celdas se monta Drive si hace falta, se prepara el dataset y después se crea o reutiliza el protocolo. El reloj de 90 minutos comienza al lanzar la creación, después de preparar los datos. No inicia entrenamientos.

In [1]:
from pathlib import Path
import base64
import hashlib
import io
import subprocess
import sys
import zipfile

# This cell carries code/configuration only. No images, weights or results.
# Editing src/configs locally requires rebuilding this snapshot with
# scripts/package_colab_notebooks.ps1 before running a remote notebook.
_CODE_BUNDLE_SHA256 = '3536b33ee1c48b59551e40befc5f14cd3a94b3bf60877a71ed1eef87d5c50b63'
_CODE_BUNDLE = '...'
REPO_ROOT = None  # Optional explicit checkout path; normally leave automatic.
INSTALL_MISSING_NOTEBOOK_DEPS = True
RUN_SETUP = False  # Full training dependencies; no experiments are launched.
RUN_MOUNT_DRIVE = False
DATASET_DRIVE_FOLDER_URL = 'https://drive.google.com/drive/folders/1vgxmmpi5zclSXRo_y2GwNIkx35QFqr5t'
DATASET_ROOT = '/content/spacelab_dataset'
STORAGE = '/content/drive/MyDrive/SpaceLAB_experiments/session_01'

if REPO_ROOT is not None:
    REPO_ROOT = Path(REPO_ROOT).expanduser().resolve()
elif sys.platform != 'win32' and Path('/content').is_dir():
    # Version-specific directory: never overwrite an older checkout or dataset.
    REPO_ROOT = Path('/content/SpaceLAB_runtime') / _CODE_BUNDLE_SHA256[:16]
else:
    REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p/'src/spacelab/notebook_setup.py').is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError('Abra el notebook dentro de SpaceLAB o conecte el kernel Colab.')

if sys.platform != 'win32' and Path('/content/SpaceLAB_runtime') == REPO_ROOT.parent:
    _payload = base64.b64decode(_CODE_BUNDLE)
    if hashlib.sha256(_payload).hexdigest() != _CODE_BUNDLE_SHA256:
        raise ValueError('El paquete de código está incompleto o alterado; recargue el notebook.')
    REPO_ROOT.mkdir(parents=True, exist_ok=True)
    _root = REPO_ROOT.resolve()
    with zipfile.ZipFile(io.BytesIO(_payload)) as _archive:
        for _entry in _archive.infolist():
            _target = (_root / _entry.filename).resolve()
            if _root not in _target.parents:
                raise ValueError('Ruta fuera del paquete de código: ' + _entry.filename)
            _contents = _archive.read(_entry)
            if _target.exists():
                if _target.read_bytes() != _contents:
                    raise RuntimeError('Código remoto editado en ' + str(_target)
                                       + '. Consérvelo y configure REPO_ROOT explícitamente.')
            else:
                _target.parent.mkdir(parents=True, exist_ok=True)
                _target.write_bytes(_contents)
    del _payload

if not (REPO_ROOT/'src/spacelab/notebook_setup.py').is_file():
    raise FileNotFoundError('REPO_ROOT no contiene el paquete actualizado: ' + str(REPO_ROOT))
print('Código del proyecto disponible en:', REPO_ROOT)
if RUN_MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if RUN_SETUP:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT/'requirements.txt')])
sys.path.insert(0, str(REPO_ROOT/'src'))
# Reload the complete project package when selecting a different snapshot.
# Run the configuration and action cells again after this bootstrap.
import importlib
_loaded = sys.modules.get('spacelab')
if _loaded is not None and Path(_loaded.__file__).resolve().parent != (REPO_ROOT/'src/spacelab').resolve():
    for _module_name in list(sys.modules):
        if _module_name == 'spacelab' or _module_name.startswith('spacelab.'):
            del sys.modules[_module_name]
    importlib.invalidate_caches()
    print('SpaceLAB actualizado en este kernel. Continúe ejecutando las celdas en orden.', flush=True)
from spacelab.notebook_setup import prepare_notebook
prepare_notebook(install_missing=INSTALL_MISSING_NOTEBOOK_DEPS)
from spacelab.time_budget import launch
from spacelab.splits import load_protocol
from spacelab.experiment_registry import config_from, collect, freeze_selection
from spacelab.io import read_json
from IPython.display import display, Markdown
PROTOCOL_PATH = Path(STORAGE)/'protocol.json'
RUN_TRAINING = False
RUN_EVALUATION = False
RUN_BENCHMARK = False
RUN_SPLITS = False
ENABLE_OPTIONAL = False
SELECTED_RUN = None
RUN_PATH = None
VALIDATION_RUN = None
RESUME_CHECKPOINT = None

Código del proyecto disponible en: c:\Users\jlaur\SpaceLAB
Python del kernel: c:\Users\jlaur\AppData\Local\Programs\Python\Python312\python.exe
Kernel local de Windows. Para usar la T4: Seleccionar kernel > Colab > servidor T4. La instalación de dependencias no cambia el kernel.
Dependencias básicas del notebook disponibles.


## Configuración editable y protocolo compartido

In [2]:
# Activado por petición del equipo. La preparación se realiza ANTES del reloj.
RUN_SPLITS = True
AUTO_PREPARE_DATA = True
GROUPS_CSV = None  # CSV id,group,verified si existen grupos verificados.
DATASET_SOURCE = '/content/drive/MyDrive/spacelab_dataset'
SOURCE_ZIP = None  # Se detecta automáticamente si hay un único ZIP en el origen.
print('Destino único:', PROTOCOL_PATH)
print('Creación de splits activada; primero se comprobarán Drive y los datos.')

Destino único: \content\drive\MyDrive\SpaceLAB_experiments\session_01\protocol.json
Creación de splits activada; primero se comprobarán Drive y los datos.


## Acción explícita, presupuesto y guardado de resultados

In [3]:
if RUN_SPLITS:
    # BEGIN PREPARATION COMPATIBILITY PATCH
    # This implementation travels with this cell, independently of old imports.
    from pathlib import Path
    import base64
    import hashlib
    import importlib.util
    import inspect
    import sys
    if sys.platform == 'win32' or not Path('/content').is_dir():
        raise RuntimeError('Seleccione el kernel Colab del mismo servidor T4.')
    _preparation_source = base64.b64decode('IiIiRXhwbGljaXQgZnV0dXJlIGRhdGEgc3RhZ2luZy4gTmV2ZXIgZG93bmxvYWRzIGEgc2hhcmVkIERyaXZlIGZvbGRlciBvbiBpbXBvcnQuIiIiCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc2h1dGlsCmltcG9ydCB0aW1lCmltcG9ydCB1dWlkCmltcG9ydCB6aXBmaWxlCmltcG9ydCB6bGliCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKZnJvbSAuaW8gaW1wb3J0IGRhdGFzZXRfcm9vdAoKRFJJVkVfRk9MREVSX1VSTCA9ICJodHRwczovL2RyaXZlLmdvb2dsZS5jb20vZHJpdmUvZm9sZGVycy8xdmd4bW1waTV6Y2xTWFJvX3kyR3dOSWt4MzVRRnFyNXQiCgoKY2xhc3MgX1Byb2dyZXNzOgogICAgIiIiUmVhbCBmaWxlL2J5dGUgcHJvZ3Jlc3MsIHBsdXMgYSBzbWFsbCBzdGF0dXMgZmlsZSBvdXRzaWRlIHRoZSBkYXRhc2V0LiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGRlc3RpbmF0aW9uLCBwaGFzZSwgdG90YWxfZmlsZXM9Tm9uZSwgdG90YWxfYnl0ZXM9Tm9uZSk6CiAgICAgICAgc2VsZi5wYXRoID0gZGVzdGluYXRpb24ucGFyZW50LyhkZXN0aW5hdGlvbi5uYW1lKycucHJvZ3Jlc3MuanNvbicpCiAgICAgICAgc2VsZi5waGFzZSA9IHBoYXNlCiAgICAgICAgc2VsZi50b3RhbF9maWxlcywgc2VsZi50b3RhbF9ieXRlcyA9IHRvdGFsX2ZpbGVzLCB0b3RhbF9ieXRlcwogICAgICAgIHNlbGYuZmlsZXMsIHNlbGYuYnl0ZXMsIHNlbGYucmV1c2VkID0gMCwgMCwgMAogICAgICAgIHNlbGYuc3RhcnRlZCwgc2VsZi5sYXN0X3dyaXRlID0gdGltZS5tb25vdG9uaWMoKSwgMAogICAgICAgIHNlbGYuYmFycyA9IFtdCiAgICAgICAgdHJ5OgogICAgICAgICAgICBmcm9tIHRxZG0uYXV0byBpbXBvcnQgdHFkbQogICAgICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICAgICAgcGFzcwogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuYmFycy5hcHBlbmQodHFkbSh0b3RhbD10b3RhbF9maWxlcywgZGVzYz1waGFzZSsnIMK3IGFyY2hpdm9zJywgdW5pdD0nYXJjaGl2bycsIG1pbmludGVydmFsPTEpKQogICAgICAgICAgICBpZiB0b3RhbF9ieXRlcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIHNlbGYuYmFycy5hcHBlbmQodHFkbSh0b3RhbD10b3RhbF9ieXRlcywgZGVzYz1waGFzZSsnIMK3IGRhdG9zJywgdW5pdD0nQicsIHVuaXRfc2NhbGU9VHJ1ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1bml0X2Rpdmlzb3I9MTAyNCwgbWluaW50ZXJ2YWw9MSkpCiAgICAgICAgcHJpbnQocGhhc2UrJyDigJQgcHJvZ3Jlc28gcmVhbCBkZSBhcmNoaXZvcyB5IGJ5dGVzOyBlbCByZWxvaiBleHBlcmltZW50YWwgbm8gaGEgY29tZW56YWRvLicsZmx1c2g9VHJ1ZSkKICAgICAgICBzZWxmLndyaXRlKGZvcmNlPVRydWUpCgogICAgZGVmIHdyaXRlKHNlbGYsIGZvcmNlPUZhbHNlLCBzdGF0ZT0nZW4gY3Vyc28nLCBjdXJyZW50PU5vbmUpOgogICAgICAgIG5vdyA9IHRpbWUubW9ub3RvbmljKCkKICAgICAgICBpZiBub3QgZm9yY2UgYW5kIG5vdy1zZWxmLmxhc3Rfd3JpdGUgPCAxOgogICAgICAgICAgICByZXR1cm4KICAgICAgICBlbGFwc2VkID0gbWF4KDAsbm93LXNlbGYuc3RhcnRlZCkKICAgICAgICBwZXJjZW50ID0gMTAwKnNlbGYuYnl0ZXMvc2VsZi50b3RhbF9ieXRlcyBpZiBzZWxmLnRvdGFsX2J5dGVzIGVsc2UgTm9uZQogICAgICAgIHNuYXBzaG90ID0geydmYXNlJzpzZWxmLnBoYXNlLCdlc3RhZG8nOnN0YXRlLCdhcmNoaXZvc19jb21wbGV0YWRvcyc6c2VsZi5maWxlcywKICAgICAgICAgICAgJ2FyY2hpdm9zX3RvdGFsJzpzZWxmLnRvdGFsX2ZpbGVzLCdieXRlc19jb3BpYWRvcyc6c2VsZi5ieXRlcywnYnl0ZXNfdG90YWwnOnNlbGYudG90YWxfYnl0ZXMsCiAgICAgICAgICAgICdwb3JjZW50YWplX2J5dGVzJzpwZXJjZW50LCdzZWd1bmRvc190cmFuc2N1cnJpZG9zJzpyb3VuZChlbGFwc2VkLDEpLCdhcmNoaXZvX2FjdHVhbCc6Y3VycmVudH0KICAgICAgICBzbmFwc2hvdFsnYXJjaGl2b3NfcmV1dGlsaXphZG9zJ10gPSBzZWxmLnJldXNlZAogICAgICAgIHNlbGYucGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICAgICAgdGVtcG9yYXJ5ID0gc2VsZi5wYXRoLndpdGhfbmFtZShzZWxmLnBhdGgubmFtZSsnLicrdXVpZC51dWlkNCgpLmhleCsnLnRtcCcpCiAgICAgICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyhzbmFwc2hvdCxlbnN1cmVfYXNjaWk9RmFsc2UsaW5kZW50PTIpLGVuY29kaW5nPSd1dGYtOCcpCiAgICAgICAgb3MucmVwbGFjZSh0ZW1wb3Jhcnksc2VsZi5wYXRoKQogICAgICAgIGlmIG5vdCBzZWxmLmJhcnM6CiAgICAgICAgICAgIHRvdGFsID0gc3RyKHNlbGYudG90YWxfZmlsZXMpIGlmIHNlbGYudG90YWxfZmlsZXMgaXMgbm90IE5vbmUgZWxzZSAnPycKICAgICAgICAgICAgcGVyY2VudGFnZSA9IGYnIHwge3BlcmNlbnQ6LjFmfSUnIGlmIHBlcmNlbnQgaXMgbm90IE5vbmUgZWxzZSAnJwogICAgICAgICAgICBwcmludChmJ3tzZWxmLnBoYXNlfToge3NlbGYuZmlsZXN9L3t0b3RhbH0gYXJjaGl2b3N7cGVyY2VudGFnZX0gfCB7c2VsZi5ieXRlcy8xMDI0KioyOi4xZn0gTWlCJyxmbHVzaD1UcnVlKQogICAgICAgIHNlbGYubGFzdF93cml0ZSA9IG5vdwoKICAgIGRlZiBhZHZhbmNlKHNlbGYsIGJ5dGVfY291bnQ9MCwgZmlsZV9jb3VudD0wLCBjdXJyZW50PU5vbmUpOgogICAgICAgIHNlbGYuYnl0ZXMgKz0gYnl0ZV9jb3VudAogICAgICAgIHNlbGYuZmlsZXMgKz0gZmlsZV9jb3VudAogICAgICAgIGlmIHNlbGYuYmFyczoKICAgICAgICAgICAgc2VsZi5iYXJzWzBdLnVwZGF0ZShmaWxlX2NvdW50KQogICAgICAgICAgICBpZiBsZW4oc2VsZi5iYXJzKT4xOgogICAgICAgICAgICAgICAgc2VsZi5iYXJzWzFdLnVwZGF0ZShieXRlX2NvdW50KQogICAgICAgIHNlbGYud3JpdGUoY3VycmVudD1jdXJyZW50KQoKICAgIGRlZiBjbG9zZShzZWxmLCBzdGF0ZSk6CiAgICAgICAgc2VsZi53cml0ZShmb3JjZT1UcnVlLHN0YXRlPXN0YXRlKQogICAgICAgIGZvciBiYXIgaW4gc2VsZi5iYXJzOgogICAgICAgICAgICBiYXIuY2xvc2UoKQoKCmRlZiBfY29weV9zdHJlYW0ocmVhZGVyLCB0YXJnZXQsIHByb2dyZXNzLCBuYW1lKToKICAgIHRhcmdldC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPVRydWUpCiAgICBwYXJ0aWFsID0gdGFyZ2V0LndpdGhfbmFtZSh0YXJnZXQubmFtZSsnLicrdXVpZC51dWlkNCgpLmhleCsnLnBhcnQnKQogICAgd2l0aCBwYXJ0aWFsLm9wZW4oJ3diJykgYXMgd3JpdGVyOgogICAgICAgIGZvciBjaHVuayBpbiBpdGVyKGxhbWJkYTogcmVhZGVyLnJlYWQoMTAyNCoxMDI0KSxiJycpOgogICAgICAgICAgICB3cml0ZXIud3JpdGUoY2h1bmspCiAgICAgICAgICAgIHByb2dyZXNzLmFkdmFuY2UoYnl0ZV9jb3VudD1sZW4oY2h1bmspLGN1cnJlbnQ9bmFtZSkKICAgIG9zLnJlcGxhY2UocGFydGlhbCx0YXJnZXQpCiAgICBwcm9ncmVzcy5hZHZhbmNlKGZpbGVfY291bnQ9MSxjdXJyZW50PW5hbWUpCgoKZGVmIHZlcmlmeV9sYXlvdXQocGF0aCk6CiAgICByb290ID0gZGF0YXNldF9yb290KHBhdGgpCiAgICBmb3Igc3BsaXQsZXhwZWN0ZWQgaW4gWygidHJhaW4iLDU5MjYpLCgidGVzdCIsODAxKSwoIm9jdWx0byIsMTc0MyldOgogICAgICAgIGNvdW50ID0gbGVuKGxpc3QoKHJvb3QvImltYWdlcyIvc3BsaXQpLmdsb2IoIiouanBnIikpKQogICAgICAgIGlmIGNvdW50ICE9IGV4cGVjdGVkOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQ29waWEgaW5jb21wbGV0YToge3NwbGl0fSB0aWVuZSB7Y291bnR9LCBzZSBlc3BlcmFiYW4ge2V4cGVjdGVkfS4gTm8gc2Ugc29icmVzY3JpYmUgYXV0b23DoXRpY2FtZW50ZS4iKQogICAgICAgIGlmIHNwbGl0ICE9ICdvY3VsdG8nOgogICAgICAgICAgICBpbWFnZV9pZHMgPSB7cC5zdGVtIGZvciBwIGluIChyb290LydpbWFnZXMnL3NwbGl0KS5nbG9iKCcqLmpwZycpfQogICAgICAgICAgICBsYWJlbF9pZHMgPSB7cC5zdGVtIGZvciBwIGluIChyb290LydsYWJlbHMnL3NwbGl0KS5nbG9iKCcqLnR4dCcpfQogICAgICAgICAgICBpZiBpbWFnZV9pZHMgIT0gbGFiZWxfaWRzOgogICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmJ0NvcGlhIGluY29tcGxldGE6IGZhbHRhbiBvIHNvYnJhbiBldGlxdWV0YXMgZGUge3NwbGl0fS4nKQogICAgcmV0dXJuIHJvb3QKCgpkZWYgc3RhZ2VfZGF0YXNldChzb3VyY2UsIGRlc3RpbmF0aW9uLCBzb3VyY2VfemlwPU5vbmUsIHJlc3VtZT1GYWxzZSk6CiAgICAiIiJDb3B5IGZyb20gYSBtb3VudGVkIERyaXZlIHNob3J0Y3V0LCBvciBleHRyYWN0IGEgdXNlci1zZWxlY3RlZCBsb2NhbCBaSVAuCgogICAgUnVuIEJFRk9SRSBvcGVuaW5nIHRoZSB0aW1lZCBzZXNzaW9uLiBUaGlzIGlzIGRhdGEgcHJlcGFyYXRpb24sIG5vdCBhCiAgICBiZW5jaG1hcmsuIE9yaWdpbmFscyBzdGF5IGludGFjdC4gcmVzdW1lPVRydWUgcmVwYWlycyBhIHBhcnRpYWwgbG9jYWwgY29weQogICAgd2l0aG91dCBkZWxldGluZyBpdC4gRXhpc3RpbmcgZmlsZXMgYXJlIHJldXNlZCBvbmx5IHdpdGggbWF0Y2hpbmcgbWV0YWRhdGEKICAgIChkaXJlY3RvcnkgY29weSkgb3IgYSB2ZXJpZmllZCBDUkMzMiAoWklQIGV4dHJhY3Rpb24pLgogICAgIiIiCiAgICBzb3VyY2UsZGVzdGluYXRpb24gPSBQYXRoKHNvdXJjZSkuZXhwYW5kdXNlcigpLFBhdGgoZGVzdGluYXRpb24pLmV4cGFuZHVzZXIoKQogICAgaWYgZGVzdGluYXRpb24uZXhpc3RzKCkgYW5kIG5vdCByZXN1bWU6CiAgICAgICAgcmV0dXJuIHZlcmlmeV9sYXlvdXQoZGVzdGluYXRpb24pCiAgICBpZiBzb3VyY2VfemlwOgogICAgICAgICMgRXh0cmFjdCBvbmx5IGludG8gYSBmcmVzaCBkaXJlY3RvcnkgYW5kIHJlamVjdCBwYXRoIHRyYXZlcnNhbC4KICAgICAgICB3aXRoIHppcGZpbGUuWmlwRmlsZShzb3VyY2VfemlwKSBhcyBhcmNoaXZlOgogICAgICAgICAgICBmb3IgbWVtYmVyIGluIGFyY2hpdmUuaW5mb2xpc3QoKToKICAgICAgICAgICAgICAgIHRhcmdldCA9IChkZXN0aW5hdGlvbi9tZW1iZXIuZmlsZW5hbWUpLnJlc29sdmUoKQogICAgICAgICAgICAgICAgaWYgZGVzdGluYXRpb24ucmVzb2x2ZSgpIG5vdCBpbiB0YXJnZXQucGFyZW50cyBhbmQgdGFyZ2V0ICE9IGRlc3RpbmF0aW9uLnJlc29sdmUoKToKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiUnV0YSBmdWVyYSBkZWwgZGVzdGlubyBlbiBaSVA6IHttZW1iZXIuZmlsZW5hbWV9IikKICAgICAgICAgICAgbWVtYmVycyA9IFttZW1iZXIgZm9yIG1lbWJlciBpbiBhcmNoaXZlLmluZm9saXN0KCkgaWYgbm90IG1lbWJlci5pc19kaXIoKV0KICAgICAgICAgICAgZGVzdGluYXRpb24ubWtkaXIocGFyZW50cz1UcnVlLGV4aXN0X29rPXJlc3VtZSkKICAgICAgICAgICAgcHJvZ3Jlc3MgPSBfUHJvZ3Jlc3MoZGVzdGluYXRpb24sJ0V4dHJheWVuZG8gWklQJyxsZW4obWVtYmVycyksc3VtKG0uZmlsZV9zaXplIGZvciBtIGluIG1lbWJlcnMpKQogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBmb3IgbWVtYmVyIGluIG1lbWJlcnM6CiAgICAgICAgICAgICAgICAgICAgdGFyZ2V0ID0gZGVzdGluYXRpb24vbWVtYmVyLmZpbGVuYW1lCiAgICAgICAgICAgICAgICAgICAgaWYgcmVzdW1lIGFuZCB0YXJnZXQuaXNfZmlsZSgpIGFuZCB0YXJnZXQuc3RhdCgpLnN0X3NpemUgPT0gbWVtYmVyLmZpbGVfc2l6ZToKICAgICAgICAgICAgICAgICAgICAgICAgY3JjID0gMAogICAgICAgICAgICAgICAgICAgICAgICB3aXRoIHRhcmdldC5vcGVuKCdyYicpIGFzIGV4aXN0aW5nOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGNodW5rIGluIGl0ZXIobGFtYmRhOiBleGlzdGluZy5yZWFkKDEwMjQqMTAyNCksYicnKToKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBjcmMgPSB6bGliLmNyYzMyKGNodW5rLGNyYykKICAgICAgICAgICAgICAgICAgICAgICAgaWYgY3JjICYgMHhmZmZmZmZmZiA9PSBtZW1iZXIuQ1JDOgogICAgICAgICAgICAgICAgICAgICAgICAgICAgcHJvZ3Jlc3MucmV1c2VkICs9IDEKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHByb2dyZXNzLmFkdmFuY2UoYnl0ZV9jb3VudD1tZW1iZXIuZmlsZV9zaXplLGZpbGVfY291bnQ9MSxjdXJyZW50PW1lbWJlci5maWxlbmFtZSkKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgICAgICAgICAgd2l0aCBhcmNoaXZlLm9wZW4obWVtYmVyKSBhcyByZWFkZXI6CiAgICAgICAgICAgICAgICAgICAgICAgIF9jb3B5X3N0cmVhbShyZWFkZXIsdGFyZ2V0LHByb2dyZXNzLG1lbWJlci5maWxlbmFtZSkKICAgICAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwcm9ncmVzcy5jbG9zZSgnaW50ZXJydW1waWRvJykKICAgICAgICAgICAgICAgIHJhaXNlCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBwcm9ncmVzcy5jbG9zZSgnY29waWEgdGVybWluYWRhOyB2ZXJpZmljYW5kbyBlc3RydWN0dXJhJykKICAgIGVsc2U6CiAgICAgICAgcm9vdCA9IGRhdGFzZXRfcm9vdChzb3VyY2UpCiAgICAgICAgaWYgcm9vdC5yZXNvbHZlKCkgPT0gZGVzdGluYXRpb24ucmVzb2x2ZSgpIG9yIHJvb3QucmVzb2x2ZSgpIGluIGRlc3RpbmF0aW9uLnJlc29sdmUoKS5wYXJlbnRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCdFbCBkZXN0aW5vIGRlIGNvcGlhIGRlYmUgZXN0YXIgZnVlcmEgZGUgbGEgY2FycGV0YSBvcmlnaW5hbC4nKQogICAgICAgIGludmVudG9yeSA9IF9Qcm9ncmVzcyhkZXN0aW5hdGlvbiwnQ29udGFuZG8gYXJjaGl2b3MgZGUgRHJpdmUnKQogICAgICAgIHRyeToKICAgICAgICAgICAgZmlsZXMgPSBbXQogICAgICAgICAgICBmb3IgcGF0aCBpbiByb290LnJnbG9iKCcqJyk6CiAgICAgICAgICAgICAgICBpZiBwYXRoLmlzX2ZpbGUoKToKICAgICAgICAgICAgICAgICAgICBzdGF0ID0gcGF0aC5zdGF0KCkKICAgICAgICAgICAgICAgICAgICBmaWxlcy5hcHBlbmQoKHBhdGgsc3RhdC5zdF9zaXplLHN0YXQuc3RfbXRpbWVfbnMpKQogICAgICAgICAgICAgICAgICAgIGludmVudG9yeS5hZHZhbmNlKGZpbGVfY291bnQ9MSxjdXJyZW50PXBhdGgubmFtZSkKICAgICAgICBleGNlcHQgQmFzZUV4Y2VwdGlvbjoKICAgICAgICAgICAgaW52ZW50b3J5LmNsb3NlKCdpbnRlcnJ1bXBpZG8nKQogICAgICAgICAgICByYWlzZQogICAgICAgIGludmVudG9yeS5jbG9zZSgnaW52ZW50YXJpbyB0ZXJtaW5hZG8nKQogICAgICAgIGRlc3RpbmF0aW9uLm1rZGlyKHBhcmVudHM9VHJ1ZSxleGlzdF9vaz1yZXN1bWUpCiAgICAgICAgcHJvZ3Jlc3MgPSBfUHJvZ3Jlc3MoZGVzdGluYXRpb24sJ0NvcGlhbmRvIERyaXZlIGEgQ29sYWInLGxlbihmaWxlcyksc3VtKHNpemUgZm9yIF8sc2l6ZSxfIGluIGZpbGVzKSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIGZvciBwYXRoLHNpemUsbXRpbWVfbnMgaW4gZmlsZXM6CiAgICAgICAgICAgICAgICByZWxhdGl2ZSA9IHBhdGgucmVsYXRpdmVfdG8ocm9vdCkKICAgICAgICAgICAgICAgIHRhcmdldCA9IGRlc3RpbmF0aW9uL3JlbGF0aXZlCiAgICAgICAgICAgICAgICBpZiByZXN1bWUgYW5kIHRhcmdldC5pc19maWxlKCk6CiAgICAgICAgICAgICAgICAgICAgc3RhdCA9IHRhcmdldC5zdGF0KCkKICAgICAgICAgICAgICAgICAgICBpZiBzdGF0LnN0X3NpemUgPT0gc2l6ZSBhbmQgc3RhdC5zdF9tdGltZV9ucyA9PSBtdGltZV9uczoKICAgICAgICAgICAgICAgICAgICAgICAgcHJvZ3Jlc3MucmV1c2VkICs9IDEKICAgICAgICAgICAgICAgICAgICAgICAgcHJvZ3Jlc3MuYWR2YW5jZShieXRlX2NvdW50PXNpemUsZmlsZV9jb3VudD0xLGN1cnJlbnQ9cmVsYXRpdmUuYXNfcG9zaXgoKSkKICAgICAgICAgICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgICAgIHdpdGggcGF0aC5vcGVuKCdyYicpIGFzIHJlYWRlcjoKICAgICAgICAgICAgICAgICAgICBfY29weV9zdHJlYW0ocmVhZGVyLHRhcmdldCxwcm9ncmVzcyxyZWxhdGl2ZS5hc19wb3NpeCgpKQogICAgICAgICAgICAgICAgc2h1dGlsLmNvcHlzdGF0KHBhdGgsdGFyZ2V0KQogICAgICAgIGV4Y2VwdCBCYXNlRXhjZXB0aW9uOgogICAgICAgICAgICBwcm9ncmVzcy5jbG9zZSgnaW50ZXJydW1waWRvJykKICAgICAgICAgICAgcmFpc2UKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcm9ncmVzcy5jbG9zZSgnY29waWEgdGVybWluYWRhOyB2ZXJpZmljYW5kbyBlc3RydWN0dXJhJykKICAgIHRyeToKICAgICAgICByZXN1bHQgPSB2ZXJpZnlfbGF5b3V0KGRlc3RpbmF0aW9uKQogICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgcHJvZ3Jlc3MuY2xvc2UoJ3ZlcmlmaWNhY2nDs24gZmFsbGlkYScpCiAgICAgICAgcmFpc2UKICAgIHByb2dyZXNzLmNsb3NlKCdjb21wbGV0YWRvIHkgdmVyaWZpY2FkbycpCiAgICBwcmludCgnRGF0YXNldCBsaXN0bzonLHJlc3VsdCxmbHVzaD1UcnVlKQogICAgcmV0dXJuIHJlc3VsdAo=')
    _preparation_sha = '057cdc3545b620b94005fe39c609fdb165462bfd1a6c3b2336e0f2c733079090'
    if hashlib.sha256(_preparation_source).hexdigest() != _preparation_sha:
        raise RuntimeError('Parche de preparación incompleto; recargue el notebook desde disco.')
    _patch_directory = Path('/content/SpaceLAB_runtime_patches')/_preparation_sha[:16]
    _patch_directory.mkdir(parents=True, exist_ok=True)
    _patch_file = _patch_directory/'preparation.py'
    if _patch_file.exists():
        if _patch_file.read_bytes() != _preparation_source:
            raise RuntimeError('El parche remoto fue editado. No se sobrescribe automáticamente.')
    else:
        _patch_file.write_bytes(_preparation_source)
    _patch_name = 'spacelab._preparation_resume_'+_preparation_sha[:16]
    if _patch_name not in sys.modules:
        _spec = importlib.util.spec_from_file_location(_patch_name, _patch_file)
        _module = importlib.util.module_from_spec(_spec)
        sys.modules[_patch_name] = _module
        try:
            _spec.loader.exec_module(_module)
        except BaseException:
            sys.modules.pop(_patch_name, None)
            raise
    _preparation = sys.modules[_patch_name]
    stage_dataset = _preparation.stage_dataset
    verify_layout = _preparation.verify_layout
    if 'resume' not in inspect.signature(stage_dataset).parameters:
        raise RuntimeError('No se cargó la preparación que admite reanudación.')
    print('Preparación con progreso y reanudación cargada en este kernel.', flush=True)
    # END PREPARATION COMPATIBILITY PATCH
    from spacelab.io import dataset_root

    if sys.platform == 'win32' or not Path('/content').is_dir():
        raise RuntimeError('Seleccione el kernel Colab del servidor T4 antes de ejecutar esta celda.')

    resolved_dataset = None
    if Path(DATASET_ROOT).exists():
        try:
            resolved_dataset = verify_layout(DATASET_ROOT)
        except ValueError:
            print('Copia local incompleta: se retomará sin borrarla.', flush=True)
    # Prefer the ZIP uploaded from this PC; preparation stays outside the budget.
    uploaded_zip = Path('/content/spacelab_dataset.zip')
    if resolved_dataset is None and not uploaded_zip.is_file():
        candidates = sorted(
            p for pattern in ('*/spacelab_dataset.zip', '*/*/spacelab_dataset.zip')
            for p in Path('/content').glob(pattern)
            if 'drive' not in p.relative_to('/content').parts and p.is_file()
        )
        if len(candidates) == 1:
            uploaded_zip = candidates[0]
        elif len(candidates) > 1:
            raise RuntimeError('Hay varios ZIP locales: ' + str(candidates))
        else:
            print('Contenido de /content:', [p.name for p in Path('/content').iterdir()], flush=True)
            raise FileNotFoundError(
                'No aparece spacelab_dataset.zip en este servidor. '
                'Termine Upload to Colab al mismo servidor del notebook. '
                'Se detiene la preparacion para evitar volver a copiar desde Drive.'
            )
    print('ZIP localizado:', uploaded_zip, flush=True)
    if resolved_dataset is None and uploaded_zip.is_file():
        expected_zip_bytes = 1835778796
        actual_zip_bytes = uploaded_zip.stat().st_size
        if actual_zip_bytes != expected_zip_bytes:
            raise RuntimeError(
                f'ZIP incompleto o distinto: {actual_zip_bytes:,} de '
                f'{expected_zip_bytes:,} bytes esperados. Espere a que termine la subida.'
            )
        print('ZIP local completo. Extrayendo con progreso; el reloj no ha comenzado.', flush=True)
        resolved_dataset = stage_dataset(
            uploaded_zip, DATASET_ROOT, source_zip=str(uploaded_zip), resume=True
        )

    if resolved_dataset is None and AUTO_PREPARE_DATA:
        if not Path('/content/drive/MyDrive').is_dir():
            from google.colab import drive
            drive.mount('/content/drive')
        source = Path(DATASET_SOURCE)
        selected_zip = SOURCE_ZIP
        if selected_zip is None and source.is_file() and source.suffix.lower() == '.zip':
            selected_zip = str(source)
        if selected_zip is None and source.is_dir():
            try:
                dataset_root(source)
            except ValueError:
                archives = sorted(source.glob('*.zip'))
                if len(archives) == 1:
                    selected_zip = str(archives[0])
                elif len(archives) > 1:
                    raise ValueError('Hay varios ZIP en el origen; indique el correcto en SOURCE_ZIP.')
        if not source.exists() and selected_zip is None:
            raise FileNotFoundError(
                'Drive está montado, pero falta el acceso directo: ' + str(source)
                + '. Añada la carpeta compartida a Mi unidad con nombre spacelab_dataset. '
                'Todavía no se ha iniciado el reloj de experimentación.'
            )
        print('Preparando datos locales; todavía no se inicia el reloj de 90 minutos...', flush=True)
        resolved_dataset = stage_dataset(source, DATASET_ROOT, source_zip=selected_zip, resume=True)
    elif resolved_dataset is None:
        raise FileNotFoundError('Dataset local ausente. Active AUTO_PREPARE_DATA.')

    DATASET_ROOT = str(resolved_dataset)
    print('Dataset listo:', DATASET_ROOT, flush=True)
    if PROTOCOL_PATH.exists():
        protocol = load_protocol(PROTOCOL_PATH, resolved_dataset)
        print('Protocolo existente reutilizado:', protocol['protocol_id'])
        display(protocol['distributions'])
    else:
        print('Ahora comienza el presupuesto compartido de 90 minutos.', flush=True)
        result = launch('splits', {'dataset': DATASET_ROOT, 'groups_csv': GROUPS_CSV}, STORAGE, cap_seconds=300)
        display(result)
        if result['status'] != 'completado':
            log = Path(result['run'])/'worker.log' if result.get('run') else None
            if log and log.exists():
                print(log.read_text(encoding='utf-8', errors='replace')[-6000:])
            raise RuntimeError('Los splits no se completaron; revise la salida anterior antes de entrenar.')
        protocol = load_protocol(PROTOCOL_PATH, resolved_dataset)
        print('Protocolo creado:', protocol['protocol_id'])
        display(protocol['distributions'])
else:
    print('Creación de splits desactivada.')

RuntimeError: Seleccione el kernel Colab del mismo servidor T4.

## Interpretación y conclusiones

No hay resultados precalculados. Revisar `result.json`, `worker.log`, las métricas y las
figuras del directorio UUID devuelto. Un estado interrumpido, omitido o early_stopped
no es un entrenamiento completo con las épocas previstas. No comparar fases, splits,
batches o cohortes distintos como una prueba controlada.

Para diagnóstico, la diferencia entre pérdida de train y val no es una medida universal
de sobreajuste: las transformaciones y el muestreo de propuestas pueden ser diferentes.
Las mejoras pequeñas requieren repeticiones futuras con otras semillas y presupuesto
propio; esta sesión con semilla 42 no estima significancia estadística.

Fundamento metodológico: Elgendy, caps. 4, 6 y 7; Lakshmanan et al., caps. 3–8.
Referencias, hipótesis y limitaciones: `EXPERIMENTAL_DESIGN.md`. No se atribuyen a los
libros los hiperparámetros concretos. El puntaje siempre usa el evaluador oficial.